In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Bronze Processing.....

In [0]:
catalog = "ete"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"
data_source = "orders"

# Need to Update S3 paths to match your actual bucket structure
base_path = f"s3a://S3 Path/{data_source}"
landing_path = f"{base_path}/landing"
processed_path = f"{base_path}/processed"


In [0]:
df_bronze = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(f"{processed_path}/*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

print(f"Total Rows Ingested: {df_bronze.count()}")
display(df_bronze.limit(10))

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"

df_bronze.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)

print(f" Data written to {bronze_table}")

In [0]:
files = dbutils.fs.ls(landing_path)
moved_count = 0
for file_info in files:
    dbutils.fs.mv(
        file_info.path, f"{processed_path}/{file_info.name}",
        True
    )
    moved_count += 1

print(f" {moved_count} files successfully moved to the processed directory!")

### Silver Processing....

In [0]:
df_orders = spark.table(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
df_orders = (
    df_orders
    .filter(F.col("order_qty").isNotNull()) 
    
    .withColumn("customer_id", F.col("customer_id").cast("string"))
    .withColumn("product_id", F.col("product_id").cast("string"))
    .withColumn("order_placement_date", F.col("order_placement_date").cast("string"))
    
    .withColumn(
        "order_placement_date",
        F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "") 
    )
    
    .withColumn(
        "order_placement_date",
        F.coalesce(
            F.try_to_date(F.col("order_placement_date"), "yyyy-MM-dd"),  
            F.try_to_date(F.col("order_placement_date"), "yyyy/MM/dd"), 
            F.try_to_date(F.col("order_placement_date"), "dd-MM-yyyy"), 
            F.try_to_date(F.col("order_placement_date"), "dd/MM/yyyy"), 
            F.try_to_date(F.col("order_placement_date"), "MMMM dd, yyyy") 
        )
    )
    
    .dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"]) 
)

In [0]:
df_products = spark.table(f"{catalog}.{silver_schema}.products")

df_joined = df_orders.join(
    df_products, 
    df_orders["product_id"] == df_products["product_code"], 
    how="inner" 
).select(
    df_orders["order_id"],
    df_orders["order_placement_date"],
    df_orders["customer_id"],
    df_orders["product_id"].alias("product_code"),
    df_orders["order_qty"],
    df_orders["read_timestamp"],
    df_orders["file_name"],
    df_orders["file_size"]
)

print("Cleaned Silver Orders:")
display(df_joined.limit(5))

In [0]:
silver_table = f"{catalog}.{silver_schema}.{data_source}" 

if not spark.catalog.tableExists(silver_table): 
    df_joined.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite") \
        .saveAsTable(silver_table)
    print(f" Created new Silver table: {silver_table}")
else:
    silver_delta = DeltaTable.forName(spark, silver_table) 
    
    silver_delta.alias("silver").merge(
        df_joined.alias("bronze"), 
        "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id" 
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute() 
    
    print(f" Merged new files into existing Silver table: {silver_table}")

### Gold Processing....

In [0]:
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}" 

df_gold = spark.sql(f"""
    SELECT 
        order_id, 
        order_placement_date as date, 
        customer_id as customer_code, 
        product_code, 
        order_qty as sold_quantity 
    FROM {silver_table}
""")

print("Prepared Child Gold Data:")
display(df_gold.limit(5))

In [0]:
if not spark.catalog.tableExists(gold_table): 
    df_gold.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .saveAsTable(gold_table) 
    print(f" Created Child Gold table: {gold_table}")
else:
    gold_delta = DeltaTable.forName(spark, gold_table) 
    gold_delta.alias("source").merge(
        df_gold.alias("gold"), 
        "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code" 
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute() 
    print(f" Merged into Child Gold table: {gold_table}")

In [0]:
df_child = spark.table(gold_table) 
df_monthly = (
    df_child
    .withColumn("month_start", F.trunc("date", "MM"))
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )
    .withColumnRenamed("month_start", "date")
)

print("Monthly Aggregated Data for Parent Merge:")
display(df_monthly.limit(5))

In [0]:
parent_fact_table = f"{catalog}.{gold_schema}.fact_orders" 

gold_parent_delta = DeltaTable.forName(spark, parent_fact_table) 

gold_parent_delta.alias("parent_gold").merge(
    df_monthly.alias("child_gold"), 
    "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code" 
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
